In [2]:
import os
import time
import requests
import pandas as pd
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()
token = os.getenv("GITHUB_TOKEN")
HEADERS={
    "Accept": "application/vnd.github+json"
}
if token:
    HEADERS["Authorization"]=f"Bearer {token}"
BASE_URL = "https://api.github.com"
FULL_NAME = "pandas-dev/pandas"
repo_url = f"{BASE_URL}/repos/{FULL_NAME}"

In [3]:
def fetch_json(url, params=None, headers=HEADERS, sleep_sec=0.2):
    res = requests.get(url, headers=headers, params=params)
    res.raise_for_status()
    time.sleep(sleep_sec)
    return res.json()

In [4]:
repo_data = fetch_json(repo_url)
repo_data

{'id': 858127,
 'node_id': 'MDEwOlJlcG9zaXRvcnk4NTgxMjc=',
 'name': 'pandas',
 'full_name': 'pandas-dev/pandas',
 'private': False,
 'owner': {'login': 'pandas-dev',
  'id': 21206976,
  'node_id': 'MDEyOk9yZ2FuaXphdGlvbjIxMjA2OTc2',
  'avatar_url': 'https://avatars.githubusercontent.com/u/21206976?v=4',
  'gravatar_id': '',
  'url': 'https://api.github.com/users/pandas-dev',
  'html_url': 'https://github.com/pandas-dev',
  'followers_url': 'https://api.github.com/users/pandas-dev/followers',
  'following_url': 'https://api.github.com/users/pandas-dev/following{/other_user}',
  'gists_url': 'https://api.github.com/users/pandas-dev/gists{/gist_id}',
  'starred_url': 'https://api.github.com/users/pandas-dev/starred{/owner}{/repo}',
  'subscriptions_url': 'https://api.github.com/users/pandas-dev/subscriptions',
  'organizations_url': 'https://api.github.com/users/pandas-dev/orgs',
  'repos_url': 'https://api.github.com/users/pandas-dev/repos',
  'events_url': 'https://api.github.com/users/

### **[1차 필터링]**

1. **중복 및 파생 변수 제거**
   - `watchers`, `forks`, `open_issues` 등은 각각 `*_count` 변수와 의미가 중복되므로 제거
   - 동일 정보를 반복하는 feature를 제거하여 데이터 중복성 감소

2. **내부 관리용 필드 제거**
   - `id`, `node_id`, `permissions`, `temp_clone_token`, `custom_properties`
   - GitHub 내부 식별 및 권한 관리용 데이터로 분석 목적과 무관

3. **단순 URL 및 API 템플릿 필드 제거**
   - `url`, `forks_url`, `keys_url`, `teams_url`, `hooks_url`, `contents_url`, `compare_url` 등
   - `{}` 형태의 템플릿 URL 포함
   - 리소스 접근용 정보일 뿐 분석 feature로서 의미 없음

4. **저수준 시스템 / 버전관리 관련 필드 제거**
   - `git_url`, `ssh_url`, `clone_url`, `svn_url`, `mirror_url`
   - 인프라 및 저장소 관리용 정보로 분석 활용도가 낮음

5. **분석 목적과 연관성이 낮은 필드 제거**
   - `is_template`, `web_commit_signoff_required`, `pull_request_creation_policy`, `visibility`, `default_branch`
   - repository 활동성, 인기, 운영 상태 분석과 직접적인 관련성 부족

6. **비정형 텍스트 데이터 제외**
   - `description`, `topics`
   - 추가적인 NLP 전처리가 필요하므로 본 단계에서는 제외

7. **결측 가능성이 높은 일부 필드 제외 (선별 적용)**
   - 일부 repository에서 값이 없는 필드들
   - 분석 안정성을 고려하여 제한적으로 유지 또는 제거



In [5]:
selected_keys = [

    # 기본 정보
    "fork",
    "html_url",
    "homepage",


    # 시간
    "created_at",
    "updated_at",
    "pushed_at",

    # 인기 / 규모
    "size",
    "stargazers_count",
    "forks_count",
    "subscribers_count",
    "open_issues_count",
    "network_count",

    # 상태 / 운영
    "has_issues",
    "has_projects",
    "has_downloads",
    "has_wiki",
    "has_pages",
    "has_discussions",
    "archived",
    "disabled",
    "allow_forking",
    "has_pull_requests",

    # nested (flatten 대상)
    "owner.login",
    "owner.type",
    "license.key",
    "license.name",
    "license.spdx_id",
    "organization.login",
    "organization.type",

    # endpoint (확장 수집용)
    "contributors_url",
    "commits_url",
    "issues_url",
    "issue_comment_url",
    "pulls_url",
    "releases_url",
    "languages_url",
    "branches_url",
    "tags_url",
    "events_url",
    "issue_events_url",
    "comments_url",
    "labels_url",
    "milestones_url",
    "deployments_url",
    "stargazers_url",
    "subscribers_url",
]

In [6]:
filtered_repo = {k: repo_data[k] for k in selected_keys if k in repo_data}
filtered_repo

{'fork': False,
 'html_url': 'https://github.com/pandas-dev/pandas',
 'homepage': 'https://pandas.pydata.org',
 'created_at': '2010-08-24T01:37:33Z',
 'updated_at': '2026-04-28T10:37:26Z',
 'pushed_at': '2026-04-28T02:35:44Z',
 'size': 390274,
 'stargazers_count': 48597,
 'forks_count': 19881,
 'subscribers_count': 1113,
 'open_issues_count': 3428,
 'network_count': 19881,
 'has_issues': True,
 'has_projects': True,
 'has_downloads': True,
 'has_wiki': False,
 'has_pages': False,
 'has_discussions': False,
 'archived': False,
 'disabled': False,
 'allow_forking': True,
 'has_pull_requests': True,
 'contributors_url': 'https://api.github.com/repos/pandas-dev/pandas/contributors',
 'commits_url': 'https://api.github.com/repos/pandas-dev/pandas/commits{/sha}',
 'issues_url': 'https://api.github.com/repos/pandas-dev/pandas/issues{/number}',
 'issue_comment_url': 'https://api.github.com/repos/pandas-dev/pandas/issues/comments{/number}',
 'pulls_url': 'https://api.github.com/repos/pandas-dev

**raw, url key 분류**

In [8]:
raw_key = {k: v for k, v in filtered_repo.items()
           if not (isinstance(v, str) and "http" in v)}
raw_key

{'fork': False,
 'created_at': '2010-08-24T01:37:33Z',
 'updated_at': '2026-04-28T10:37:26Z',
 'pushed_at': '2026-04-28T02:35:44Z',
 'size': 390274,
 'stargazers_count': 48597,
 'forks_count': 19881,
 'subscribers_count': 1113,
 'open_issues_count': 3428,
 'network_count': 19881,
 'has_issues': True,
 'has_projects': True,
 'has_downloads': True,
 'has_wiki': False,
 'has_pages': False,
 'has_discussions': False,
 'archived': False,
 'disabled': False,
 'allow_forking': True,
 'has_pull_requests': True}

In [9]:
url_key = {k: v for k, v in filtered_repo.items()
           if (isinstance(v, str) and "http" in v)}
url_key

{'html_url': 'https://github.com/pandas-dev/pandas',
 'homepage': 'https://pandas.pydata.org',
 'contributors_url': 'https://api.github.com/repos/pandas-dev/pandas/contributors',
 'commits_url': 'https://api.github.com/repos/pandas-dev/pandas/commits{/sha}',
 'issues_url': 'https://api.github.com/repos/pandas-dev/pandas/issues{/number}',
 'issue_comment_url': 'https://api.github.com/repos/pandas-dev/pandas/issues/comments{/number}',
 'pulls_url': 'https://api.github.com/repos/pandas-dev/pandas/pulls{/number}',
 'releases_url': 'https://api.github.com/repos/pandas-dev/pandas/releases{/id}',
 'languages_url': 'https://api.github.com/repos/pandas-dev/pandas/languages',
 'branches_url': 'https://api.github.com/repos/pandas-dev/pandas/branches{/branch}',
 'tags_url': 'https://api.github.com/repos/pandas-dev/pandas/tags',
 'events_url': 'https://api.github.com/repos/pandas-dev/pandas/events',
 'issue_events_url': 'https://api.github.com/repos/pandas-dev/pandas/issues/events{/number}',
 'comm

**JSON 문자열 → Python dict/list로 파싱됨 → 구조 기반 순회**

In [ ]:
from pathlib import Path

def is_github_api_url(value):
    return isinstance(value, str) and value.startswith("https://api.github.com/")

def is_template_url(value):
    return isinstance(value, str) and ("{" in value and "}" in value)

def save_progress(rows, error_log, save_prefix="walk_json_progress"):
    pd.DataFrame(rows).to_csv(f"sample/{save_prefix}_rows.csv", index=False)
    pd.DataFrame(error_log).to_csv(f"sample/{save_prefix}_errors.csv", index=False)

def walk_json(
    obj,
    parent_path="",
    depth=0,
    max_depth=1,
    visited_urls=None,
    error_log=None,
    rows=None,
    verbose=True,
    save_every=100,
    counter=None,
    save_prefix="walk_json_progress"
):
    if visited_urls is None:
        visited_urls = set()
    if error_log is None:
        error_log = []
    if rows is None:
        rows = []
    if counter is None:
        counter = {"n": 0}

    # depth 제한
    if depth > max_depth:
        return rows, error_log

    def append_row(row):
        rows.append(row)
        counter["n"] += 1

        if save_every and counter["n"] % save_every == 0:
            save_progress(rows, error_log, save_prefix)
            if verbose:
                print(f"[checkpoint] saved {counter['n']} rows")

    if isinstance(obj, dict):
        for key, value in obj.items():
            current_path = f"{parent_path}.{key}" if parent_path else key

            append_row({
                "path": current_path,
                "key": key,
                "dtype": type(value).__name__,
                "example_value": str(value),
                "is_github_api_url": is_github_api_url(value),
                "fetch_status": None,
                "error_message": None,
                "depth": depth,
            })

            if isinstance(value, (dict, list)):
                walk_json(
                    value,
                    parent_path=current_path,
                    depth=depth + 1,
                    max_depth=max_depth,
                    visited_urls=visited_urls,
                    error_log=error_log,
                    rows=rows,
                    verbose=verbose,
                    save_every=save_every,
                    counter=counter,
                    save_prefix=save_prefix
                )

            elif (
                depth < max_depth
                and is_github_api_url(value)
                and not is_template_url(value)
                and value not in visited_urls
            ):
                visited_urls.add(value)

                try:
                    sub_data = fetch_json(value)

                    append_row({
                        "path": current_path + ".__api_fetch__",
                        "key": "__api_fetch__",
                        "dtype": type(sub_data).__name__,
                        "example_value": str(value),
                        "is_github_api_url": True,
                        "fetch_status": "success",
                        "error_message": None,
                        "depth": depth,
                    })

                    if sub_data is not None:
                        walk_json(
                            sub_data,
                            parent_path=current_path + ".__api__",
                            depth=depth + 1,
                            max_depth=max_depth,
                            visited_urls=visited_urls,
                            error_log=error_log,
                            rows=rows,
                            verbose=verbose,
                            save_every=save_every,
                            counter=counter,
                            save_prefix=save_prefix
                        )

                except requests.exceptions.HTTPError as e:
                    msg = f"HTTPError: {e}"
                    error_log.append({
                        "path": current_path,
                        "url": value,
                        "error_type": "HTTPError",
                        "error_message": str(e),
                        "depth": depth,
                    })

                    append_row({
                        "path": current_path + ".__api_error__",
                        "key": "__api_error__",
                        "dtype": "error",
                        "example_value": str(value),
                        "is_github_api_url": True,
                        "fetch_status": "failed",
                        "error_message": msg,
                        "depth": depth,
                    })

                    if verbose:
                        print(f"[HTTPError] {current_path} -> {value}")
                        print(f"  {e}")

                except Exception as e:
                    msg = f"{type(e).__name__}: {e}"
                    error_log.append({
                        "path": current_path,
                        "url": value,
                        "error_type": type(e).__name__,
                        "error_message": str(e),
                        "depth": depth,
                    })

                    append_row({
                        "path": current_path + ".__api_error__",
                        "key": "__api_error__",
                        "dtype": "error",
                        "example_value": str(value),
                        "is_github_api_url": True,
                        "fetch_status": "failed",
                        "error_message": msg,
                        "depth": depth,
                    })

                    if verbose:
                        print(f"[Error] {current_path} -> {value}")
                        print(f"  {msg}")

    elif isinstance(obj, list):
        for idx, item in enumerate(obj):
            current_path = f"{parent_path}[{idx}]"

            append_row({
                "path": current_path,
                "key": f"[{idx}]",
                "dtype": type(item).__name__,
                "example_value": str(item),
                "is_github_api_url": is_github_api_url(item),
                "fetch_status": None,
                "error_message": None,
                "depth": depth,
            })

            if isinstance(item, (dict, list)):
                walk_json(
                    item,
                    parent_path=current_path,
                    depth=depth + 1,
                    max_depth=max_depth,
                    visited_urls=visited_urls,
                    error_log=error_log,
                    rows=rows,
                    verbose=verbose,
                    save_every=save_every,
                    counter=counter,
                    save_prefix=save_prefix
                )

            elif (
                depth < max_depth
                and is_github_api_url(item)
                and not is_template_url(item)
                and item not in visited_urls
            ):
                visited_urls.add(item)

                try:
                    sub_data = fetch_json(item)

                    append_row({
                        "path": current_path + ".__api_fetch__",
                        "key": "__api_fetch__",
                        "dtype": type(sub_data).__name__,
                        "example_value": str(item),
                        "is_github_api_url": True,
                        "fetch_status": "success",
                        "error_message": None,
                        "depth": depth,
                    })

                    if sub_data is not None:
                        walk_json(
                            sub_data,
                            parent_path=current_path + ".__api__",
                            depth=depth + 1,
                            max_depth=max_depth,
                            visited_urls=visited_urls,
                            error_log=error_log,
                            rows=rows,
                            verbose=verbose,
                            save_every=save_every,
                            counter=counter,
                            save_prefix=save_prefix
                        )

                except requests.exceptions.HTTPError as e:
                    msg = f"HTTPError: {e}"
                    error_log.append({
                        "path": current_path,
                        "url": item,
                        "error_type": "HTTPError",
                        "error_message": str(e),
                        "depth": depth,
                    })

                    append_row({
                        "path": current_path + ".__api_error__",
                        "key": "__api_error__",
                        "dtype": "error",
                        "example_value": str(item),
                        "is_github_api_url": True,
                        "fetch_status": "failed",
                        "error_message": msg,
                        "depth": depth,
                    })

                    if verbose:
                        print(f"[HTTPError] {current_path} -> {item}")
                        print(f"  {e}")

                except Exception as e:
                    msg = f"{type(e).__name__}: {e}"
                    error_log.append({
                        "path": current_path,
                        "url": item,
                        "error_type": type(e).__name__,
                        "error_message": str(e),
                        "depth": depth,
                    })

                    append_row({
                        "path": current_path + ".__api_error__",
                        "key": "__api_error__",
                        "dtype": "error",
                        "example_value": str(item),
                        "is_github_api_url": True,
                        "fetch_status": "failed",
                        "error_message": msg,
                        "depth": depth,
                    })

                    if verbose:
                        print(f"[Error] {current_path} -> {item}")
                        print(f"  {msg}")

    # 마지막에도 저장
    if depth == 0:
        save_progress(rows, error_log, save_prefix)
        if verbose:
            print(f"[done] total rows={len(rows)}, total errors={len(error_log)}")

    return rows, error_log

In [55]:
print(len(raw_key))
print(len(url_key))

20
19


In [56]:
total_key = {**raw_key, **url_key}
repo_data = {k: v for k, v in total_key.items()}

In [57]:
repo_data

{'fork': False,
 'created_at': '2010-08-24T01:37:33Z',
 'updated_at': '2026-04-28T10:37:26Z',
 'pushed_at': '2026-04-28T02:35:44Z',
 'size': 390274,
 'stargazers_count': 48597,
 'forks_count': 19881,
 'subscribers_count': 1113,
 'open_issues_count': 3428,
 'network_count': 19881,
 'has_issues': True,
 'has_projects': True,
 'has_downloads': True,
 'has_wiki': False,
 'has_pages': False,
 'has_discussions': False,
 'archived': False,
 'disabled': False,
 'allow_forking': True,
 'has_pull_requests': True,
 'html_url': 'https://github.com/pandas-dev/pandas',
 'homepage': 'https://pandas.pydata.org',
 'contributors_url': 'https://api.github.com/repos/pandas-dev/pandas/contributors',
 'commits_url': 'https://api.github.com/repos/pandas-dev/pandas/commits{/sha}',
 'issues_url': 'https://api.github.com/repos/pandas-dev/pandas/issues{/number}',
 'issue_comment_url': 'https://api.github.com/repos/pandas-dev/pandas/issues/comments{/number}',
 'pulls_url': 'https://api.github.com/repos/pandas-dev

In [58]:
flattened_data, error_log = walk_json(repo_data)

[checkpoint] saved 100 rows
[checkpoint] saved 200 rows
[done] total rows=215, total errors=0


In [64]:
df = pd.DataFrame(flattened_data)
df

,path,key,dtype,example_value,is_github_api_url,fetch_status,error_message,depth
0,fork,fork,bool,False,False,NaN,None,0
1,created_at,created_at,str,2010-08-24T01:37:33Z,False,NaN,None,0
2,updated_at,updated_at,str,2026-04-28T10:37:26Z,False,NaN,None,0
3,pushed_at,pushed_at,str,2026-04-28T02:35:44Z,False,NaN,None,0
4,size,size,int,390274,False,NaN,None,0
...,...,...,...,...,...,...,...,...
210,subscribers_url.__api__[25],[25],dict,"{'login': 'invinciblejha', 'id': 1061207, 'nod...",False,NaN,None,1
211,subscribers_url.__api__[26],[26],dict,"{'login': 'stephenwlin', 'id': 2342637, 'node_...",False,NaN,None,1
212,subscribers_url.__api__[27],[27],dict,"{'login': 'strayhorn', 'id': 3350114, 'node_id...",False,NaN,None,1
213,subscribers_url.__api__[28],[28],dict,"{'login': 'binaryechoes', 'id': 1815971, 'node...",False,NaN,None,1


---

### Nested GitHub API Flatten 결과 정리

---

### 정리 목적

현재 `walk_json`을 통해 flatten한 결과에는 repository 자체의 health를 직접 설명하는 정보와,
nested endpoint에서 파생된 개별 객체 정보가 혼합되어 있다.

최종 목표는 **repository 단위의 health feature table 구성**이므로,
모든 flattened row를 그대로 유지하는 것은 비효율적이다.

따라서 각 데이터는 아래 3가지 기준으로 구분한다.

* 그대로 유지
* 집계 후 사용
* 제거

---

### 최종 분류 기준표

| 데이터 그룹             | 예시 path / 형태                                                        | 처리 방식   | 이유               | 활용 방식              |
| ------------------ | ------------------------------------------------------------------- | ------- | ---------------- | ------------------ |
| Repository root 통계 | stargazers_count, forks_count, subscribers_count, open_issues_count | 유지      | repo-level 정량 정보 | 직접 feature 사용      |
| Repository 상태 flag | has_issues, has_wiki, archived, disabled, allow_forking             | 유지      | 운영 상태 설명         | governance feature |
| 시간 정보              | created_at, updated_at, pushed_at                                   | 유지      | 활동성/신선도 반영       | time-based feature |
| endpoint URL       | contributors_url, languages_url 등                                   | 집계 후 사용 | 추가 데이터 source    | fetch 후 feature 생성 |
| contributor 객체     | contributors_url.**api**[i]                                         | 집계 후 사용 | 개별 user row는 불필요 | 분포/집중도 계산          |
| contributor 프로필    | login, id, avatar_url 등                                             | 제거      | 식별자 수준 정보        | 사용 안 함             |
| language raw       | languages_url.**api**.Python 등                                      | 집계 후 사용 | 언어 구성 의미 있음      | entropy / 비율       |
| tag raw            | tags_url.**api**[i]                                                 | 집계 후 사용 | release 정보 포함    | maturity feature   |
| tag 내부 URL         | zipball_url 등                                                       | 제거      | 다운로드 링크          | 의미 없음              |
| event raw          | events_url.**api**[i]                                               | 집계 후 사용 | 활동 유형 중요         | type count         |
| event actor        | actor.login 등                                                       | 제거      | 개인 식별 정보         | 의미 없음              |
| deployment raw     | deployments_url.**api**[i]                                          | 집계 후 사용 | 존재 여부만 중요        | count/flag         |
| stargazer raw      | stargazers_url.**api**[i]                                           | 제거      | count로 대체 가능     | 제거                 |
| subscriber raw     | subscribers_url.**api**[i]                                          | 제거      | count로 충분        | 제거                 |
| URL/템플릿            | commits_url, contents_url 등                                         | 제거      | navigation용      | 제거                 |
| 식별자                | id, node_id                                                         | 제거      | 의미 없음            | 제거                 |

---

### 그대로 유지해야 하는 데이터

| 유형         | 예시                            | 이유      |
| ---------- | ----------------------------- | ------- |
| popularity | stargazers_count, forks_count | 외부 관심도  |
| activity   | open_issues_count             | 운영 상태   |
| governance | has_issues, has_discussions   | 협업 가능성  |
| status     | archived, disabled            | 프로젝트 상태 |
| time       | created_at, pushed_at         | 유지보수 상태 |

---

### 집계 후 사용해야 하는 데이터

| 그룹           | raw 데이터                  | 이유          | 생성 feature             |
| ------------ | ------------------------ | ----------- | ---------------------- |
| contributors | contributors_url.**api** | 구조 분석 필요    | num_contributors, gini |
| languages    | languages_url.**api**    | 기술 스택 구성    | entropy, top_ratio     |
| tags         | tags_url.**api**         | release 성숙도 | stable_ratio           |
| events       | events_url.**api**       | 협업 활동       | event_ratio            |
| deployments  | deployments_url.**api**  | 배포 여부       | has_deployments        |

---

### 제거 대상 데이터

| 대상         | 예시                      | 이유         |
| ---------- | ----------------------- | ---------- |
| 사용자 정보     | login, avatar_url       | 개인 식별용     |
| 링크         | html_url, followers_url | feature 아님 |
| 다운로드 URL   | zipball_url             | 의미 없음      |
| 템플릿 URL    | commits_url{/sha}       | 구조용        |
| raw 객체 row | **api**[i]              | 단위 불일치     |

---

### 현재 데이터에 대한 직접 판단

| 데이터                         | 처리      |
| --------------------------- | ------- |
| contributors_url.**api**[i] | 제거 + 집계 |
| contributors_url            | 유지      |
| languages_url.**api**       | 집계      |
| tags_url.**api**            | 집계      |
| events_url.**api**          | 집계      |
| deployments_url.**api**     | 집계      |
| stargazers_url.**api**      | 제거      |
| subscribers_url.**api**     | 제거      |

---

### 최종 원칙

| 원칙              | 설명                    |
| --------------- | --------------------- |
| repo 단위 유지      | 1 row = 1 repo        |
| raw → aggregate | 객체 대신 통계              |
| URL 제거          | 링크는 feature 아님        |
| 분포 중심           | count, ratio, entropy |
| 범용성 유지          | 모든 repo에 적용 가능        |

---

### 핵심 결론

> 개별 객체(contributor, event, tag)는 제거하고, repository-level 통계로 변환하여 사용한다.

---

### **[2차 필터링]**

불필요한 root / template 제거

In [65]:
DROP_PATHS = [
    # --- URL / LINK ---
    "html_url",
    "homepage",

    # --- TEMPLATE API URL (endpoint 아님, template 형태라 useless) ---
    "commits_url",
    "issues_url",
    "issue_comment_url",
    "pulls_url",
    "releases_url",
    "branches_url",
    "issue_events_url",
    "comments_url",
    "labels_url",
    "milestones_url",

    # --- IDENTIFIER ---
    "id",
    "node_id",
    "gravatar_id",
]

In [66]:
df.shape

(215, 8)

In [67]:
df.head()

,path,key,dtype,example_value,is_github_api_url,fetch_status,error_message,depth
0,fork,fork,bool,False,False,NaN,None,0
1,created_at,created_at,str,2010-08-24T01:37:33Z,False,NaN,None,0
2,updated_at,updated_at,str,2026-04-28T10:37:26Z,False,NaN,None,0
3,pushed_at,pushed_at,str,2026-04-28T02:35:44Z,False,NaN,None,0
4,size,size,int,390274,False,NaN,None,0


In [68]:
df2 = df[~df["path"].isin(DROP_PATHS)].copy()

In [69]:
df2.shape

(203, 8)

In [70]:
df2

,path,key,dtype,example_value,is_github_api_url,fetch_status,error_message,depth
0,fork,fork,bool,False,False,NaN,None,0
1,created_at,created_at,str,2010-08-24T01:37:33Z,False,NaN,None,0
2,updated_at,updated_at,str,2026-04-28T10:37:26Z,False,NaN,None,0
3,pushed_at,pushed_at,str,2026-04-28T02:35:44Z,False,NaN,None,0
4,size,size,int,390274,False,NaN,None,0
...,...,...,...,...,...,...,...,...
210,subscribers_url.__api__[25],[25],dict,"{'login': 'invinciblejha', 'id': 1061207, 'nod...",False,NaN,None,1
211,subscribers_url.__api__[26],[26],dict,"{'login': 'stephenwlin', 'id': 2342637, 'node_...",False,NaN,None,1
212,subscribers_url.__api__[27],[27],dict,"{'login': 'strayhorn', 'id': 3350114, 'node_id...",False,NaN,None,1
213,subscribers_url.__api__[28],[28],dict,"{'login': 'binaryechoes', 'id': 1815971, 'node...",False,NaN,None,1


In [72]:
df2.to_csv("sample/2nd-filtered.csv", index=False)

In [73]:
df2.head(10)

,path,key,dtype,example_value,is_github_api_url,fetch_status,error_message,depth
0,fork,fork,bool,False,False,NaN,None,0
1,created_at,created_at,str,2010-08-24T01:37:33Z,False,NaN,None,0
2,updated_at,updated_at,str,2026-04-28T10:37:26Z,False,NaN,None,0
3,pushed_at,pushed_at,str,2026-04-28T02:35:44Z,False,NaN,None,0
4,size,size,int,390274,False,NaN,None,0
5,stargazers_count,stargazers_count,int,48597,False,NaN,None,0
6,forks_count,forks_count,int,19881,False,NaN,None,0
7,subscribers_count,subscribers_count,int,1113,False,NaN,None,0
8,open_issues_count,open_issues_count,int,3428,False,NaN,None,0
9,network_count,network_count,int,19881,False,NaN,None,0


---

**nested -> feature로 변환 -> 제거**

### **[Contributors]**

**contributors data 형태**

```raw
  {
    "login": "jbrockmendel",
    "id": 8078968,
    "node_id": "MDQ6VXNlcjgwNzg5Njg=",
    "avatar_url": "https://avatars.githubusercontent.com/u/8078968?v=4",
    "gravatar_id": "",
    "url": "https://api.github.com/users/jbrockmendel",
    "html_url": "https://github.com/jbrockmendel",
    "followers_url": "https://api.github.com/users/jbrockmendel/followers",
    "following_url": "https://api.github.com/users/jbrockmendel/following{/other_user}",
    "gists_url": "https://api.github.com/users/jbrockmendel/gists{/gist_id}",
    "starred_url": "https://api.github.com/users/jbrockmendel/starred{/owner}{/repo}",
    "subscriptions_url": "https://api.github.com/users/jbrockmendel/subscriptions",
    "organizations_url": "https://api.github.com/users/jbrockmendel/orgs",
    "repos_url": "https://api.github.com/users/jbrockmendel/repos",
    "events_url": "https://api.github.com/users/jbrockmendel/events{/privacy}",
    "received_events_url": "https://api.github.com/users/jbrockmendel/received_events",
    "type": "User",
    "user_view_type": "public",
    "site_admin": false,
    "contributions": 5189
  },...
```

> **=> features to make**
> 
> - num_contributors
> - total_contributions
> - top1_contribution_share
> - top5_contribution_share
> - contribution_gini
> - median_contributions


In [80]:
import re
import numpy as np

def gini(values):
    values = np.array(values, dtype=float)
    if len(values) == 0 or values.sum() == 0:
        return 0.0
    values = np.sort(values)
    n = len(values)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * values)) / (n * values.sum()) - (n + 1) / n


def make_contributor_features(data: pd.DataFrame) -> pd.DataFrame:

    # contributors row
    contributor_rows = data[
        data["path"].str.startswith("contributors_url.__api__")
    ]

    num_contributors = len(contributor_rows)

    # regex로 contributions 추출
    def extract_contribution(s):
        if not isinstance(s, str):
            return 0
        match = re.search(r"'contributions': (\d+)", s)
        return int(match.group(1)) if match else 0

    contributions = contributor_rows["example_value"].apply(extract_contribution)

    total_contributions = contributions.sum()

    if total_contributions == 0:
        return pd.DataFrame([{
            "num_contributors": num_contributors,
            "total_contributions": 0,
            "top1_contribution_share": 0.0,
            "top5_contribution_share": 0.0,
            "contribution_gini": 0.0,
            "median_contributions": 0.0,
        }])

    contributions_sorted = contributions.sort_values(ascending=False)

    return pd.DataFrame([{
        "num_contributors": num_contributors,
        "total_contributions": int(total_contributions),
        "top1_contribution_share": contributions_sorted.iloc[0] / total_contributions,
        "top5_contribution_share": contributions_sorted.head(5).sum() / total_contributions,
        "contribution_gini": gini(contributions),
        "median_contributions": float(contributions.median()),
    }])

In [81]:
contributor_features = make_contributor_features(df2)
contributor_features

,num_contributors,total_contributions,top1_contribution_share,top5_contribution_share,contribution_gini,median_contributions
0,30,25256,0.209416,0.644243,0.596302,340.0


In [82]:
contributor_features.to_csv("sample/contributor_features.csv", index=False)

---

### **[Languages]**

> **=> features to make**
> - primary_language_ratio
> - top2_ratio
> - top3_ratio
> - entropy
> - minor_lang_ratio

    +언어를 그룹화:

| 그룹 | 예시 |
| --- | --- |
| main | Python, C, Java |
| infra | Shell, Dockerfile |
| markup | HTML, CSS |
| config/build | Meson |


> - infra_ratio
> - markup_ratio
> - is_monolingual (단일 lan인지)
> - compiled_ratio (성능 중심 프로젝트인지)

```raw
COMPILED_LANGS = 
    ["C", "C++", "Rust", "Go", "Java", "Swift"]
INTERPRETED_LANGS = 
    ["Python", "JavaScript", "Ruby", "PHP"]
SCRIPT_LANGS = 
    ["Shell"]
MARKUP_LANGS =
    ["HTML", "CSS", "XSLT", "Smarty", "Go Template"]
BUILD_LANGS = 
    ["Meson", "Makefile"]
```

In [91]:
COMPILED_LANGS = ["C", "C++", "Rust", "Go", "Java", "Swift"]
INTERPRETED_LANGS = ["Python", "JavaScript", "Ruby", "PHP"]
SCRIPT_LANGS = ["Shell"]
MARKUP_LANGS = ["HTML", "CSS", "XSLT", "Smarty", "Go Template"]
BUILD_LANGS = ["Meson", "Makefile"]

def make_language_features(data: pd.DataFrame) -> pd.DataFrame:
    lang_rows = data[
        data["path"].astype(str).str.contains(r"^languages_url\.__api__\.", regex=True, na=False)
    ].copy()

    print("matched language rows:", len(lang_rows))
    print(lang_rows[["path", "key", "example_value"]].head())

    if lang_rows.empty:
        return pd.DataFrame([{
            "primary_language_ratio": 0.0,
            "top2_ratio": 0.0,
            "top3_ratio": 0.0,
            "entropy": 0.0,
            "minor_lang_ratio": 0.0,
            "infra_ratio": 0.0,
            "markup_ratio": 0.0,
            "is_monolingual": 0,
            "compiled_ratio": 0.0,
        }])

    lang_rows["bytes"] = pd.to_numeric(lang_rows["example_value"], errors="coerce").fillna(0)

    total = lang_rows["bytes"].sum()

    if total == 0:
        return pd.DataFrame([{
            "primary_language_ratio": 0.0,
            "top2_ratio": 0.0,
            "top3_ratio": 0.0,
            "entropy": 0.0,
            "minor_lang_ratio": 0.0,
            "infra_ratio": 0.0,
            "markup_ratio": 0.0,
            "is_monolingual": 0,
            "compiled_ratio": 0.0,
        }])

    ratios = lang_rows["bytes"] / total
    sorted_ratios = ratios.sort_values(ascending=False).to_numpy()

    entropy = -np.sum(ratios * np.log(ratios + 1e-9))
    minor_lang_ratio = ratios[ratios < 0.05].sum()

    infra_ratio = lang_rows.loc[
        lang_rows["key"].isin(SCRIPT_LANGS + BUILD_LANGS),
        "bytes"
    ].sum() / total

    markup_ratio = lang_rows.loc[
        lang_rows["key"].isin(MARKUP_LANGS),
        "bytes"
    ].sum() / total

    compiled_ratio = lang_rows.loc[
        lang_rows["key"].isin(COMPILED_LANGS),
        "bytes"
    ].sum() / total

    return pd.DataFrame([{
        "primary_language_ratio": float(sorted_ratios[0]),
        "top2_ratio": float(sorted_ratios[:2].sum()),
        "top3_ratio": float(sorted_ratios[:3].sum()),
        "entropy": float(entropy),
        "minor_lang_ratio": float(minor_lang_ratio),
        "infra_ratio": float(infra_ratio),
        "markup_ratio": float(markup_ratio),
        "is_monolingual": int(len(lang_rows) == 1),
        "compiled_ratio": float(compiled_ratio),
    }])

In [92]:
language_features = make_language_features(df2)
language_features.to_csv("sample/language_features.csv", index=False)

matched language rows: 11
                            path     key example_value
61  languages_url.__api__.Python  Python      22299478
62  languages_url.__api__.Cython  Cython       1527931
63       languages_url.__api__.C       C        353600
64    languages_url.__api__.HTML    HTML        285657
65   languages_url.__api__.Meson   Meson         13824


In [93]:
language_features

,primary_language_ratio,top2_ratio,top3_ratio,entropy,minor_lang_ratio,infra_ratio,markup_ratio,is_monolingual,compiled_ratio
0,0.910064,0.972421,0.986852,0.383726,0.027579,0.000669,0.01245,0,0.01446


---